# Part 11 · Turn a REST API into MCP tools, then compose them

Most of the tools an agent needs already exist as REST APIs. Writing an MCP server for each one is the slow way round. agentgateway reads the API's OpenAPI spec and serves every operation as an MCP tool, with no code and no server to build.

That gets you one tool per endpoint. Real tasks are usually a few calls in a row, so the second half of this lab chains those calls into **new** tools on a **new** MCP endpoint: the steps, their order and the shape of the answer all live in gateway config. Finally a model is given that endpoint and left to get on with it.

This part runs on **`mesh1`** on the `ar-ingress` Gateway. The REST API is the Swagger Petstore, run in the cluster.

**What we will do**

1. **Run the Petstore** in the cluster and call it as plain REST.
2. **Turn its OpenAPI spec into MCP tools**, one per operation.
3. **Call those tools** and get real Petstore data back over MCP.
4. **Compose a workflow** into two new tools on a new MCP endpoint.
5. **Run the workflow tools** and read the result back.
6. **Hand the new endpoint to a model** and watch which tools it picks.

## 1. Run the Petstore REST API

The Swagger Petstore is the standard OpenAPI sample: pets, orders and users. It runs in its own `rest-api` namespace. Nothing here knows anything about MCP.

In [ ]:
kc apply -f demo-scripts/yaml-rest-mcp/petstore.yaml
kc -n rest-api rollout status deploy/petstore --timeout=180s

echo; echo "  == plain REST: GET /api/v3/pet/1 =="
kc get --raw /api/v1/namespaces/rest-api/services/petstore:8080/proxy/api/v3/pet/1; echo

## 2. Turn the OpenAPI spec into MCP tools

Two objects. A ConfigMap holds the spec the API already publishes, and an `EnterpriseAgentgatewayBackend` with `protocol: OpenAPI` points at it. The gateway parses the spec and derives each tool's input schema from the operation's path, query and body parameters. A route on `ar-ingress` makes it an MCP endpoint.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 220" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="720" height="220" rx="10" fill="#ffffff"/>
  <text x="360" y="28" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">OpenAPI spec in, MCP tools out</text>
  <defs><marker id="oa" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs>
  <rect x="20" y="70" width="150" height="70" rx="9" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/>
  <text x="95" y="100" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">MCP client</text>
  <text x="95" y="118" text-anchor="middle" font-size="10" fill="#475569">agent or model</text>
  <line x1="170" y1="105" x2="246" y2="105" stroke="#334155" stroke-width="1.6" marker-end="url(#oa)"/>
  <text x="208" y="95" text-anchor="middle" font-size="9.5" fill="#475569">MCP</text>
  <rect x="248" y="58" width="224" height="94" rx="10" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/>
  <text x="360" y="82" text-anchor="middle" font-size="12.5" font-weight="700" fill="#14532d">agentgateway</text>
  <text x="360" y="101" text-anchor="middle" font-size="10" fill="#166534">backend rest-petstore</text>
  <text x="360" y="117" text-anchor="middle" font-size="10" fill="#166534">protocol: OpenAPI</text>
  <text x="360" y="133" text-anchor="middle" font-size="10" fill="#166534">19 operations, 19 tools</text>
  <line x1="472" y1="105" x2="548" y2="105" stroke="#334155" stroke-width="1.6" marker-end="url(#oa)"/>
  <text x="510" y="95" text-anchor="middle" font-size="9.5" fill="#475569">REST</text>
  <rect x="550" y="70" width="150" height="70" rx="9" fill="#e2e8f0" stroke="#64748b" stroke-width="1.5"/>
  <text x="625" y="100" text-anchor="middle" font-size="12" font-weight="600" fill="#334155">Petstore API</text>
  <text x="625" y="118" text-anchor="middle" font-size="10" fill="#475569">namespace rest-api</text>
  <text x="360" y="190" text-anchor="middle" font-size="11" fill="#64748b">The spec sits in a ConfigMap. Each operation becomes a tool whose input schema is built from its parameters and body.</text>
</svg>

In [ ]:
echo "  == load the Petstore's own OpenAPI spec into a ConfigMap =="
kc get --raw /api/v1/namespaces/rest-api/services/petstore:8080/proxy/api/v3/openapi.json \
  | kc -n agentgateway-system create configmap rest-petstore-openapi --from-file=schema=/dev/stdin \
      --dry-run=client -o yaml \
  | kc apply -f -

In [ ]:
echo "  == serve it as MCP with protocol: OpenAPI =="
kc apply -f - <<EOF
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: { name: rest-petstore, namespace: agentgateway-system }
spec:
  entMcp:
    sessionRouting: Stateless          # a REST API has no MCP session to keep
    failureMode: FailClosed
    targets:
    - name: petstore
      static:
        host: petstore.rest-api.svc.cluster.local
        port: 8080
        protocol: OpenAPI              # parse the spec, one tool per operation
        openAPI:
          schemaRef: { name: rest-petstore-openapi }
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: rest-petstore, namespace: agentgateway-system }
spec:
  parentRefs: [{ name: ar-ingress }]
  hostnames: ["rest-petstore.${LB}.sslip.io"]
  rules:
  - backendRefs: [{ group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: rest-petstore }]
EOF

In [ ]:
echo "  == the REST operations are now MCP tools =="
mcp "http://rest-petstore.${LB}.sslip.io/" tools

## 3. Call the REST operations as MCP tools

Same endpoint, now used. `addPet` takes the pet as its body, `getPetById` takes the ID as a path parameter, and the gateway turns each MCP call into the matching REST request.

In [ ]:
MCP="http://rest-petstore.${LB}.sslip.io/"
echo "  -- addPet: create pet 777 --"
mcp "$MCP" call addPet '{"body":{"id":777,"name":"Demo Dog","status":"available","photoUrls":[]}}'
echo; echo "  -- getPetById: read it back --"
mcp "$MCP" call getPetById '{"path":{"petId":777}}'

## 4. Compose a workflow into a new MCP tool

One tool per endpoint still leaves the model doing the plumbing: call one, read the ID, call the next, count the results. Each of those is a model turn, and each turn re-sends the whole conversation.

A composable target (`entMcp.targets[].custom`) moves that plumbing into the gateway. It declares a new tool with its own input schema and a list of steps. A step is either an **MCP** call to a tool on another backend or a plain **HTTP** call, and every step's JSON result lands in `output.<step>` for the later steps and for the final `output` expression, which is CEL.

This backend serves two new tools on a new endpoint:

- **`onboard-pet`** registers a pet through the `addPet` MCP tool from §2, reads it back over HTTP, then counts the available pets through `findPetsByStatus`.
- **`adopt-pet`** looks a pet up, places an order for it and marks it sold: three HTTP calls, one tool.

How to read the YAML in step 1:

- Each entry under `targets` is one new tool. Its `name` is the tool name.
- `inputSchema` is what the client must send. CEL reads it as `input.*`.
- Each step is `mcp` (a tool on another backend) or `http` (a REST call). Its JSON result becomes `output.<step name>`, so `verify` builds its path from `output.add.id`, the ID the `add` step got back.
- `output` is a CEL expression over those results. It is the answer the client gets.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 300" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="760" height="300" rx="10" fill="#ffffff"/>
  <text x="380" y="28" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">One tool call in, three upstream calls out</text>
  <defs><marker id="wf" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs>
  <rect x="20" y="116" width="130" height="64" rx="9" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/>
  <text x="85" y="143" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">MCP client</text>
  <text x="85" y="161" text-anchor="middle" font-size="10" fill="#475569">onboard-pet</text>
  <line x1="150" y1="148" x2="196" y2="148" stroke="#334155" stroke-width="1.6" marker-end="url(#wf)"/>
  <rect x="198" y="52" width="344" height="210" rx="10" fill="#f5f3ff" stroke="#7c3aed" stroke-width="1.8"/>
  <text x="370" y="76" text-anchor="middle" font-size="12.5" font-weight="700" fill="#4c1d95">agentgateway: backend pet-workflow</text>
  <text x="370" y="93" text-anchor="middle" font-size="10" fill="#6d28d9">entMcp.targets[].custom</text>
  <rect x="218" y="106" width="304" height="34" rx="7" fill="#ede9fe" stroke="#a78bfa"/>
  <text x="370" y="128" text-anchor="middle" font-size="10.5" fill="#4c1d95">1  add        MCP   addPet on rest-petstore</text>
  <rect x="218" y="148" width="304" height="34" rx="7" fill="#ede9fe" stroke="#a78bfa"/>
  <text x="370" y="170" text-anchor="middle" font-size="10.5" fill="#4c1d95">2  verify    HTTP  GET /api/v3/pet/{output.add.id}</text>
  <rect x="218" y="190" width="304" height="34" rx="7" fill="#ede9fe" stroke="#a78bfa"/>
  <text x="370" y="212" text-anchor="middle" font-size="10.5" fill="#4c1d95">3  available MCP   findPetsByStatus</text>
  <text x="370" y="248" text-anchor="middle" font-size="10" fill="#6d28d9">output: a CEL expression over all three results</text>
  <line x1="542" y1="148" x2="596" y2="148" stroke="#334155" stroke-width="1.6" marker-end="url(#wf)"/>
  <rect x="598" y="116" width="140" height="64" rx="9" fill="#e2e8f0" stroke="#64748b" stroke-width="1.5"/>
  <text x="668" y="143" text-anchor="middle" font-size="12" font-weight="600" fill="#334155">Petstore API</text>
  <text x="668" y="161" text-anchor="middle" font-size="10" fill="#475569">namespace rest-api</text>
  <text x="380" y="288" text-anchor="middle" font-size="11" fill="#64748b">The client sees one tool with its own input schema. The steps, their order and the output shape are all gateway config.</text>
</svg>

In [ ]:
echo "  == compose two new MCP tools, and give them their own MCP endpoint =="
kc apply -f - <<EOF
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: { name: pet-workflow, namespace: agentgateway-system }
spec:
  entMcp:
    prefixMode: Never                  # tool names stay onboard-pet and adopt-pet
    targets:
    - name: onboard-pet
      custom:
        description: "Register a new pet, read it back from the store, and report how many pets are now available."
        inputSchema:
          type: object
          additionalProperties: false
          required: [id, name]
          properties:
            id: { type: integer, description: "Pet ID to register" }
            name: { type: string, description: "Pet name" }
        steps:
        - name: add                    # MCP step: reuse the OpenAPI tool from part 2
          mcp:
            backendRef: { group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: rest-petstore, target: petstore }
            tool: addPet
            arguments: '{"body": {"id": input.id, "name": input.name, "status": "available", "photoUrls": []}}'
        - name: verify                 # HTTP step: uses the add step's result
          http:
            backendRef: { group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: petstore-upstream }
            path: '"/api/v3/pet/" + string(output.add.id)'
        - name: available
          mcp:
            backendRef: { group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: rest-petstore, target: petstore }
            tool: findPetsByStatus
            arguments: '{"query": {"status": "available"}}'
        output: |
          {"registered": output.verify.name, "id": output.verify.id,
           "status": output.verify.status, "availableNow": size(output.available.data)}
    - name: adopt-pet
      custom:
        description: "Adopt a pet: look it up, place an order for it, and mark it sold."
        inputSchema:
          type: object
          additionalProperties: false
          required: [petId]
          properties:
            petId: { type: integer, description: "ID of the pet to adopt" }
        steps:
        - name: pet
          http:
            backendRef: { group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: petstore-upstream }
            path: '"/api/v3/pet/" + string(input.petId)'
        - name: order                  # a body makes it JSON; the gateway sets Content-Type
          http:
            backendRef: { group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: petstore-upstream }
            method: POST
            path: '"/api/v3/store/order"'
            body: '{"id": input.petId, "petId": input.petId, "quantity": 1, "status": "placed", "complete": false}'
        - name: sold
          http:
            backendRef: { group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: petstore-upstream }
            method: PUT
            path: '"/api/v3/pet"'
            body: '{"id": output.pet.id, "name": output.pet.name, "photoUrls": output.pet.photoUrls, "status": "sold"}'
        output: |
          "Adopted " + output.sold.name + " (pet " + string(output.sold.id) + "). Order "
          + string(output.order.id) + " is " + output.order.status + ", and the pet is now " + output.sold.status + "."
---
# The HTTP steps above call this: a plain backend for the REST API itself.
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: { name: petstore-upstream, namespace: agentgateway-system }
spec:
  static: { host: petstore.rest-api.svc.cluster.local, port: 8080 }
---
# The new MCP endpoint: one host, one backend, both tools.
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: pet-workflow, namespace: agentgateway-system }
spec:
  parentRefs: [{ name: ar-ingress }]
  hostnames: ["pet-workflow.${LB}.sslip.io"]
  rules:
  - backendRefs: [{ group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: pet-workflow }]
EOF

In [ ]:
echo "  == the new endpoint serves exactly the two composed tools =="
mcp "http://pet-workflow.${LB}.sslip.io/" tools

## 5. Run the workflow tools

One MCP call each. Behind `onboard-pet` the gateway makes three upstream calls, and behind `adopt-pet` another three, and the client sees neither.

In [ ]:
mcp "http://pet-workflow.${LB}.sslip.io/" call onboard-pet '{"id":5151,"name":"Nova"}'

In [ ]:
mcp "http://pet-workflow.${LB}.sslip.io/" call adopt-pet '{"petId":5151}'

echo; echo "  == the REST API agrees =="
kc get --raw /api/v1/namespaces/rest-api/services/petstore:8080/proxy/api/v3/pet/5151; echo

## 6. Hand the new endpoint to Claude

The last step puts a model in front of the endpoint. `mcp-agent.py` is the smallest possible agent, about 60 lines, and Claude does the thinking:

1. It lists the tools `pet-workflow` serves, over MCP through agentgateway.
2. It sends Claude (Claude Haiku, through the Anthropic API) the task in plain English, plus those two tool definitions.
3. Claude decides which tool to call and with what arguments. The script runs that call through agentgateway and hands Claude the result.
4. That repeats until Claude replies in words.

Claude sees two tools and nothing else: no Petstore operations, no IDs to thread between calls. The output shows each turn.

In [ ]:
python3 demo-scripts/mcp-agent.py --model claude-haiku-4-5 \
  --url "http://pet-workflow.${LB}.sslip.io/" \
  "Register a new pet called Juniper with ID 6161, then adopt her. Tell me the order number."

## Reset

**Reset lab** removes the Petstore namespace, both MCP backends, the upstream backend, the two routes and the spec ConfigMap. The `ar-ingress` Gateway stays up.